In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalogo", "bank_dev")
dbutils.widgets.text("esquema_bronze", "bronze")
dbutils.widgets.text("esquema_silver", "silver")
dbutils.widgets.text("storageName", "saprojectbankdeveastus")
dbutils.widgets.text("containerSilver", "silver")

In [0]:
catalogo         = dbutils.widgets.get("catalogo")
esquema_bronze   = dbutils.widgets.get("esquema_bronze")
esquema_silver   = dbutils.widgets.get("esquema_silver")
storageName      = dbutils.widgets.get("storageName")
containerSilver  = dbutils.widgets.get("containerSilver")

In [0]:
tabla_bronze_fraud_labels = f"{catalogo}.{esquema_bronze}.fraud_labels"
tabla_silver_fraud_labels = f"{catalogo}.{esquema_silver}.dim_fraud_labels"

In [0]:
_df = spark.read.table("bank_dev.bronze.fraud_labels")

In [0]:
# ===================== TRANSFORMACIÓN Y CARGA FRAUD LABELS (STATIC) =====================

# 1. Transformación a nivel Silver
df_silver_fraud = _df \
    .withColumn("is_fraud", F.when(F.col("target") == "Yes", True).otherwise(False)) \
    .withColumn("silver_load_date", F.current_timestamp()) \
    .drop("target", "ingestion_date") # Eliminamos el string original y la fecha de bronze

# 2. Guardado en Silver (Modo Overwrite)
print(f"Sobrescribiendo la tabla {tabla_silver_fraud_labels}...")

(df_silver_fraud.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(tabla_silver_fraud_labels))

print("Carga de Fraud Labels completada exitosamente.")